In [0]:
# Databricks notebook source

# 01 — Develop and Validate Portfolio State Engine

This notebook:

1. Validates the centralized portfolio configuration.
2. Builds deterministic development and held-out event ledgers.
3. Replays cash, order, fill, cancellation, sale, and settlement events.
4. Validates cash, cost basis, realized P&L, unrealized P&L, and total equity.
5. Proves duplicate, invalid, and conflicting events are rejected.
6. Confirms deterministic replay.
7. Writes an approved portfolio-state validation artifact.
8. Reloads the artifact for a deployment-readiness smoke test.

This notebook does not create or update production Delta tables.

In [0]:
# Add the repository root to the Python path

import sys
from pathlib import Path


def find_repository_root(
    start_path: Path,
) -> Path:
    current_path = start_path.resolve()

    for candidate in [
        current_path,
        *current_path.parents,
    ]:
        if (
            candidate
            / "src"
            / "portfolio"
        ).is_dir():
            return candidate

    raise FileNotFoundError(
        "Could not find the repository root "
        "containing src/portfolio."
    )


REPOSITORY_ROOT = find_repository_root(
    Path.cwd()
)

if str(REPOSITORY_ROOT) not in sys.path:
    sys.path.insert(
        0,
        str(REPOSITORY_ROOT),
    )

print(
    f"Repository root: {REPOSITORY_ROOT}"
)

In [0]:
# Imports and portfolio settings

from datetime import (
    datetime,
    timedelta,
    timezone,
)
from decimal import Decimal
import platform

import pandas as pd

from src.portfolio.accounting import (
    calculate_market_value,
    calculate_total_cost_basis,
    calculate_total_equity,
    calculate_total_market_value,
    calculate_total_unrealized_pnl,
    calculate_unrealized_pnl,
)
from src.portfolio.models import (
    OrderSide,
    PortfolioEvent,
    PortfolioEventType,
)
from src.portfolio.portfolio_config import (
    CONTRACT_PAYOUT,
    PORTFOLIO_ID,
    PORTFOLIO_SCHEMA_VERSION,
    PORTFOLIO_VALIDATION_ARTIFACT_PATH,
    PORTFOLIO_VALIDATION_ARTIFACT_VERSION,
    REQUIRED_PORTFOLIO_INVARIANTS,
    STARTING_CASH,
)
from src.portfolio.store import (
    read_json_artifact,
    snapshot_to_dict,
    write_json_artifact,
)
from src.portfolio.transitions import (
    apply_event,
    create_empty_snapshot,
    mark_to_market,
    replay_events,
)
from src.portfolio.validation import (
    build_portfolio_configuration_snapshot,
    evaluate_validation_case,
    raise_for_failed_validations,
    validate_event,
    validate_portfolio_configuration,
    validate_snapshot,
    validation_results_to_records,
)


print(
    f"Portfolio ID: {PORTFOLIO_ID}"
)

print(
    f"Starting cash: {STARTING_CASH}"
)

print(
    "Validation artifact path: "
    f"{PORTFOLIO_VALIDATION_ARTIFACT_PATH}"
)

In [0]:
# Validate the active portfolio configuration

validate_portfolio_configuration()

portfolio_configuration_snapshot = (
    build_portfolio_configuration_snapshot()
)

configuration_df = pd.DataFrame(
    [
        {
            "configuration_key": key,
            "configuration_value": value,
        }
        for key, value
        in portfolio_configuration_snapshot.items()
        if key != "required_invariants"
    ]
)

display(
    configuration_df
)

print(
    "Portfolio configuration passed validation."
)

In [0]:
# Create the reproducible development portfolio and cash event

DEVELOPMENT_PORTFOLIO_ID = (
    f"{PORTFOLIO_ID}_development"
)
DEVELOPMENT_TICKER = (
    "KXCFB-DEV-HOME"
)
DEVELOPMENT_OPPOSING_TICKER = (
    "KXCFB-DEV-AWAY"
)
DEVELOPMENT_ESPN_EVENT_ID = (
    "development_event_001"
)
DEVELOPMENT_BASE_TS = datetime(
    2025,
    9,
    6,
    16,
    0,
    tzinfo=timezone.utc,
)


development_empty_snapshot = (
    create_empty_snapshot(
        portfolio_id=(
            DEVELOPMENT_PORTFOLIO_ID
        ),
        as_of_ts=DEVELOPMENT_BASE_TS,
    )
)

development_initial_cash_event = (
    PortfolioEvent(
        event_id=(
            "development_cash_deposit"
        ),
        event_sequence=1,
        portfolio_id=(
            DEVELOPMENT_PORTFOLIO_ID
        ),
        event_ts=DEVELOPMENT_BASE_TS,
        event_type=(
            PortfolioEventType.CASH_DEPOSIT
        ),
        cash_amount=STARTING_CASH,
    )
)

development_initial_snapshot = apply_event(
    snapshot=development_empty_snapshot,
    event=development_initial_cash_event,
)

print(
    "Development portfolio initialized with "
    f"available cash of "
    f"{development_initial_snapshot.account.available_cash}."
)

In [0]:
# Define the development event ledger

# Fixed values in this cell are reproducible validation fixtures.
# Production accounting behavior and persistence identifiers come
# from src/portfolio/portfolio_config.py.

development_entry_events = [
    PortfolioEvent(
        event_id="development_buy_order",
        event_sequence=2,
        portfolio_id=(
            DEVELOPMENT_PORTFOLIO_ID
        ),
        event_ts=(
            DEVELOPMENT_BASE_TS
            + timedelta(minutes=1)
        ),
        event_type=(
            PortfolioEventType.ORDER_PLACED
        ),
        ticker=DEVELOPMENT_TICKER,
        espn_event_id=(
            DEVELOPMENT_ESPN_EVENT_ID
        ),
        order_id="development_buy_order_001",
        decision_id="development_decision_001",
        side=OrderSide.BUY,
        quantity=10,
        price=Decimal("0.45"),
    ),
    PortfolioEvent(
        event_id="development_buy_fill_001",
        event_sequence=3,
        portfolio_id=(
            DEVELOPMENT_PORTFOLIO_ID
        ),
        event_ts=(
            DEVELOPMENT_BASE_TS
            + timedelta(minutes=2)
        ),
        event_type=PortfolioEventType.BUY_FILL,
        ticker=DEVELOPMENT_TICKER,
        espn_event_id=(
            DEVELOPMENT_ESPN_EVENT_ID
        ),
        order_id="development_buy_order_001",
        decision_id="development_decision_001",
        side=OrderSide.BUY,
        quantity=6,
        price=Decimal("0.44"),
        fee=Decimal("0.06"),
    ),
    PortfolioEvent(
        event_id="development_buy_fill_002",
        event_sequence=4,
        portfolio_id=(
            DEVELOPMENT_PORTFOLIO_ID
        ),
        event_ts=(
            DEVELOPMENT_BASE_TS
            + timedelta(minutes=3)
        ),
        event_type=PortfolioEventType.BUY_FILL,
        ticker=DEVELOPMENT_TICKER,
        espn_event_id=(
            DEVELOPMENT_ESPN_EVENT_ID
        ),
        order_id="development_buy_order_001",
        decision_id="development_decision_001",
        side=OrderSide.BUY,
        quantity=4,
        price=Decimal("0.43"),
        fee=Decimal("0.04"),
    ),
]

development_partial_exit_events = [
    PortfolioEvent(
        event_id=(
            "development_sell_order_partial"
        ),
        event_sequence=5,
        portfolio_id=(
            DEVELOPMENT_PORTFOLIO_ID
        ),
        event_ts=(
            DEVELOPMENT_BASE_TS
            + timedelta(minutes=5)
        ),
        event_type=(
            PortfolioEventType.ORDER_PLACED
        ),
        ticker=DEVELOPMENT_TICKER,
        espn_event_id=(
            DEVELOPMENT_ESPN_EVENT_ID
        ),
        order_id="development_sell_order_001",
        decision_id="development_decision_002",
        side=OrderSide.SELL,
        quantity=4,
        price=Decimal("0.54"),
    ),
    PortfolioEvent(
        event_id=(
            "development_sell_fill_partial"
        ),
        event_sequence=6,
        portfolio_id=(
            DEVELOPMENT_PORTFOLIO_ID
        ),
        event_ts=(
            DEVELOPMENT_BASE_TS
            + timedelta(minutes=6)
        ),
        event_type=PortfolioEventType.SELL_FILL,
        ticker=DEVELOPMENT_TICKER,
        espn_event_id=(
            DEVELOPMENT_ESPN_EVENT_ID
        ),
        order_id="development_sell_order_001",
        decision_id="development_decision_002",
        side=OrderSide.SELL,
        quantity=2,
        price=Decimal("0.56"),
        fee=Decimal("0.02"),
    ),
    PortfolioEvent(
        event_id=(
            "development_sell_order_cancelled"
        ),
        event_sequence=7,
        portfolio_id=(
            DEVELOPMENT_PORTFOLIO_ID
        ),
        event_ts=(
            DEVELOPMENT_BASE_TS
            + timedelta(minutes=7)
        ),
        event_type=(
            PortfolioEventType.ORDER_CANCELLED
        ),
        order_id="development_sell_order_001",
        decision_id="development_decision_003",
    ),
]

development_final_exit_events = [
    PortfolioEvent(
        event_id="development_sell_order_final",
        event_sequence=8,
        portfolio_id=(
            DEVELOPMENT_PORTFOLIO_ID
        ),
        event_ts=(
            DEVELOPMENT_BASE_TS
            + timedelta(minutes=8)
        ),
        event_type=(
            PortfolioEventType.ORDER_PLACED
        ),
        ticker=DEVELOPMENT_TICKER,
        espn_event_id=(
            DEVELOPMENT_ESPN_EVENT_ID
        ),
        order_id="development_sell_order_002",
        decision_id="development_decision_004",
        side=OrderSide.SELL,
        quantity=8,
        price=Decimal("0.57"),
    ),
    PortfolioEvent(
        event_id="development_sell_fill_final",
        event_sequence=9,
        portfolio_id=(
            DEVELOPMENT_PORTFOLIO_ID
        ),
        event_ts=(
            DEVELOPMENT_BASE_TS
            + timedelta(minutes=9)
        ),
        event_type=PortfolioEventType.SELL_FILL,
        ticker=DEVELOPMENT_TICKER,
        espn_event_id=(
            DEVELOPMENT_ESPN_EVENT_ID
        ),
        order_id="development_sell_order_002",
        decision_id="development_decision_004",
        side=OrderSide.SELL,
        quantity=8,
        price=Decimal("0.58"),
        fee=Decimal("0.08"),
    ),
]

print(
    "Development event ledger prepared."
)

In [0]:
# Replay development events through each lifecycle stage

development_entry_snapshot = replay_events(
    snapshot=development_initial_snapshot,
    events=development_entry_events,
)

development_marked_snapshot = mark_to_market(
    snapshot=development_entry_snapshot,
    market_prices={
        DEVELOPMENT_TICKER: Decimal("0.55"),
    },
    as_of_ts=(
        DEVELOPMENT_BASE_TS
        + timedelta(minutes=4)
    ),
)

development_partial_exit_snapshot = (
    replay_events(
        snapshot=development_marked_snapshot,
        events=development_partial_exit_events,
    )
)

development_final_snapshot = replay_events(
    snapshot=development_partial_exit_snapshot,
    events=development_final_exit_events,
)

print(
    "Development event ledger replay completed."
)

In [0]:
# Inspect development account state across lifecycle stages

development_snapshots_by_stage = {
    "initialized": development_initial_snapshot,
    "entered": development_entry_snapshot,
    "marked_to_market": (
        development_marked_snapshot
    ),
    "partially_exited": (
        development_partial_exit_snapshot
    ),
    "fully_exited": development_final_snapshot,
}

development_account_rows = []

for stage_name, snapshot in (
    development_snapshots_by_stage.items()
):
    development_account_rows.append(
        {
            "stage": stage_name,
            "as_of_ts": snapshot.as_of_ts,
            "last_event_sequence": (
                snapshot.last_event_sequence
            ),
            "available_cash": float(
                snapshot.account.available_cash
            ),
            "reserved_cash": float(
                snapshot.account.reserved_cash
            ),
            "market_value": float(
                calculate_total_market_value(
                    snapshot.positions
                )
            ),
            "cost_basis": float(
                calculate_total_cost_basis(
                    snapshot.positions
                )
            ),
            "realized_pnl": float(
                snapshot.account.realized_pnl
            ),
            "unrealized_pnl": float(
                calculate_total_unrealized_pnl(
                    snapshot.positions
                )
            ),
            "total_equity": float(
                calculate_total_equity(
                    account=snapshot.account,
                    positions=snapshot.positions,
                )
            ),
            "fees_paid": float(
                snapshot.account.fees_paid
            ),
        }
    )

development_account_df = pd.DataFrame(
    development_account_rows
)

display(
    development_account_df
)

In [0]:
# Inspect development position and order state

development_position_rows = []

for stage_name, snapshot in {
    "marked_to_market": (
        development_marked_snapshot
    ),
    "partially_exited": (
        development_partial_exit_snapshot
    ),
}.items():
    for position in snapshot.positions:
        development_position_rows.append(
            {
                "stage": stage_name,
                "ticker": position.ticker,
                "espn_event_id": (
                    position.espn_event_id
                ),
                "quantity": position.quantity,
                "average_entry_price": float(
                    position.average_entry_price
                ),
                "cost_basis": float(
                    position.cost_basis
                ),
                "current_price": float(
                    position.current_price
                ),
                "market_value": float(
                    calculate_market_value(
                        quantity=position.quantity,
                        current_price=(
                            position.current_price
                        ),
                    )
                ),
                "realized_pnl": float(
                    position.realized_pnl
                ),
                "unrealized_pnl": float(
                    calculate_unrealized_pnl(
                        position
                    )
                ),
                "fees_paid": float(
                    position.fees_paid
                ),
                "opened_at": position.opened_at,
                "last_increased_at": (
                    position.last_increased_at
                ),
                "last_reduced_at": (
                    position.last_reduced_at
                ),
            }
        )

development_order_rows = [
    {
        "order_id": order.order_id,
        "ticker": order.ticker,
        "side": order.side.value,
        "requested_quantity": (
            order.requested_quantity
        ),
        "filled_quantity": (
            order.filled_quantity
        ),
        "remaining_quantity": (
            order.remaining_quantity
        ),
        "limit_price": float(
            order.limit_price
        ),
        "reserved_cash": float(
            order.reserved_cash
        ),
        "status": order.status.value,
        "decision_id": order.decision_id,
    }
    for order in development_final_snapshot.orders
]

display(
    pd.DataFrame(
        development_position_rows
    )
)

display(
    pd.DataFrame(
        development_order_rows
    )
)

In [0]:
# Validate development accounting and lifecycle outcomes

development_validation_results = [
    evaluate_validation_case(
        validation_name=(
            "development_initial_snapshot_valid"
        ),
        validation_callable=lambda: (
            validate_snapshot(
                development_initial_snapshot
            )
        ),
    ),
    evaluate_validation_case(
        validation_name=(
            "development_entry_snapshot_valid"
        ),
        validation_callable=lambda: (
            validate_snapshot(
                development_entry_snapshot
            )
        ),
    ),
    evaluate_validation_case(
        validation_name=(
            "development_marked_snapshot_valid"
        ),
        validation_callable=lambda: (
            validate_snapshot(
                development_marked_snapshot
            )
        ),
    ),
    evaluate_validation_case(
        validation_name=(
            "development_partial_exit_valid"
        ),
        validation_callable=lambda: (
            validate_snapshot(
                development_partial_exit_snapshot
            )
        ),
    ),
    evaluate_validation_case(
        validation_name=(
            "development_final_snapshot_valid"
        ),
        validation_callable=lambda: (
            validate_snapshot(
                development_final_snapshot
            )
        ),
    ),
    evaluate_validation_case(
        validation_name=(
            "development_marked_equity_expected"
        ),
        validation_callable=lambda: (
            calculate_total_equity(
                account=(
                    development_marked_snapshot.account
                ),
                positions=(
                    development_marked_snapshot.positions
                ),
            )
            == Decimal("1001.0400")
        ),
    ),
    evaluate_validation_case(
        validation_name=(
            "development_partial_quantity_expected"
        ),
        validation_callable=lambda: (
            development_partial_exit_snapshot
            .positions[0]
            .quantity
            == 8
        ),
    ),
    evaluate_validation_case(
        validation_name=(
            "development_final_cash_expected"
        ),
        validation_callable=lambda: (
            development_final_snapshot
            .account
            .available_cash
            == Decimal("1001.2000")
        ),
    ),
    evaluate_validation_case(
        validation_name=(
            "development_final_realized_pnl_expected"
        ),
        validation_callable=lambda: (
            development_final_snapshot
            .account
            .realized_pnl
            == Decimal("1.2000")
        ),
    ),
    evaluate_validation_case(
        validation_name=(
            "development_final_position_closed"
        ),
        validation_callable=lambda: (
            len(
                development_final_snapshot.positions
            )
            == 0
        ),
    ),
]

raise_for_failed_validations(
    development_validation_results
)

print(
    "Development validations passed."
)

In [0]:
# Define the held-out validation event ledger

HELD_OUT_PORTFOLIO_ID = (
    f"{PORTFOLIO_ID}_held_out"
)
HELD_OUT_TICKER = "KXCFB-HELD-HOME"
HELD_OUT_ESPN_EVENT_ID = (
    "held_out_event_001"
)
HELD_OUT_BASE_TS = datetime(
    2025,
    9,
    13,
    16,
    0,
    tzinfo=timezone.utc,
)

held_out_events = [
    PortfolioEvent(
        event_id="held_out_cash_deposit",
        event_sequence=1,
        portfolio_id=HELD_OUT_PORTFOLIO_ID,
        event_ts=HELD_OUT_BASE_TS,
        event_type=(
            PortfolioEventType.CASH_DEPOSIT
        ),
        cash_amount=STARTING_CASH,
    ),
    PortfolioEvent(
        event_id="held_out_buy_order_partial",
        event_sequence=2,
        portfolio_id=HELD_OUT_PORTFOLIO_ID,
        event_ts=(
            HELD_OUT_BASE_TS
            + timedelta(minutes=1)
        ),
        event_type=(
            PortfolioEventType.ORDER_PLACED
        ),
        ticker=HELD_OUT_TICKER,
        espn_event_id=HELD_OUT_ESPN_EVENT_ID,
        order_id="held_out_buy_order_001",
        decision_id="held_out_decision_001",
        side=OrderSide.BUY,
        quantity=12,
        price=Decimal("0.35"),
    ),
    PortfolioEvent(
        event_id="held_out_buy_fill_partial",
        event_sequence=3,
        portfolio_id=HELD_OUT_PORTFOLIO_ID,
        event_ts=(
            HELD_OUT_BASE_TS
            + timedelta(minutes=2)
        ),
        event_type=PortfolioEventType.BUY_FILL,
        ticker=HELD_OUT_TICKER,
        espn_event_id=HELD_OUT_ESPN_EVENT_ID,
        order_id="held_out_buy_order_001",
        decision_id="held_out_decision_001",
        side=OrderSide.BUY,
        quantity=5,
        price=Decimal("0.34"),
        fee=Decimal("0.05"),
    ),
    PortfolioEvent(
        event_id="held_out_buy_order_cancelled",
        event_sequence=4,
        portfolio_id=HELD_OUT_PORTFOLIO_ID,
        event_ts=(
            HELD_OUT_BASE_TS
            + timedelta(minutes=3)
        ),
        event_type=(
            PortfolioEventType.ORDER_CANCELLED
        ),
        order_id="held_out_buy_order_001",
        decision_id="held_out_decision_002",
    ),
    PortfolioEvent(
        event_id="held_out_buy_order_increase",
        event_sequence=5,
        portfolio_id=HELD_OUT_PORTFOLIO_ID,
        event_ts=(
            HELD_OUT_BASE_TS
            + timedelta(minutes=4)
        ),
        event_type=(
            PortfolioEventType.ORDER_PLACED
        ),
        ticker=HELD_OUT_TICKER,
        espn_event_id=HELD_OUT_ESPN_EVENT_ID,
        order_id="held_out_buy_order_002",
        decision_id="held_out_decision_003",
        side=OrderSide.BUY,
        quantity=3,
        price=Decimal("0.36"),
    ),
    PortfolioEvent(
        event_id="held_out_buy_fill_increase",
        event_sequence=6,
        portfolio_id=HELD_OUT_PORTFOLIO_ID,
        event_ts=(
            HELD_OUT_BASE_TS
            + timedelta(minutes=5)
        ),
        event_type=PortfolioEventType.BUY_FILL,
        ticker=HELD_OUT_TICKER,
        espn_event_id=HELD_OUT_ESPN_EVENT_ID,
        order_id="held_out_buy_order_002",
        decision_id="held_out_decision_003",
        side=OrderSide.BUY,
        quantity=3,
        price=Decimal("0.35"),
        fee=Decimal("0.03"),
    ),
    PortfolioEvent(
        event_id="held_out_sell_order_partial",
        event_sequence=7,
        portfolio_id=HELD_OUT_PORTFOLIO_ID,
        event_ts=(
            HELD_OUT_BASE_TS
            + timedelta(minutes=6)
        ),
        event_type=(
            PortfolioEventType.ORDER_PLACED
        ),
        ticker=HELD_OUT_TICKER,
        espn_event_id=HELD_OUT_ESPN_EVENT_ID,
        order_id="held_out_sell_order_001",
        decision_id="held_out_decision_004",
        side=OrderSide.SELL,
        quantity=3,
        price=Decimal("0.40"),
    ),
    PortfolioEvent(
        event_id="held_out_sell_fill_partial",
        event_sequence=8,
        portfolio_id=HELD_OUT_PORTFOLIO_ID,
        event_ts=(
            HELD_OUT_BASE_TS
            + timedelta(minutes=7)
        ),
        event_type=PortfolioEventType.SELL_FILL,
        ticker=HELD_OUT_TICKER,
        espn_event_id=HELD_OUT_ESPN_EVENT_ID,
        order_id="held_out_sell_order_001",
        decision_id="held_out_decision_004",
        side=OrderSide.SELL,
        quantity=3,
        price=Decimal("0.42"),
        fee=Decimal("0.03"),
    ),
    PortfolioEvent(
        event_id="held_out_settlement",
        event_sequence=9,
        portfolio_id=HELD_OUT_PORTFOLIO_ID,
        event_ts=(
            HELD_OUT_BASE_TS
            + timedelta(minutes=8)
        ),
        event_type=PortfolioEventType.SETTLEMENT,
        ticker=HELD_OUT_TICKER,
        espn_event_id=HELD_OUT_ESPN_EVENT_ID,
        quantity=5,
        price=CONTRACT_PAYOUT,
    ),
]

print(
    "Held-out validation ledger prepared."
)

In [0]:
# Replay and validate the held-out event ledger

held_out_empty_snapshot = create_empty_snapshot(
    portfolio_id=HELD_OUT_PORTFOLIO_ID,
    as_of_ts=HELD_OUT_BASE_TS,
)

held_out_final_snapshot = replay_events(
    snapshot=held_out_empty_snapshot,
    events=held_out_events,
)

held_out_validation_results = [
    evaluate_validation_case(
        validation_name=(
            "held_out_final_snapshot_valid"
        ),
        validation_callable=lambda: (
            validate_snapshot(
                held_out_final_snapshot
            )
        ),
    ),
    evaluate_validation_case(
        validation_name=(
            "held_out_final_cash_expected"
        ),
        validation_callable=lambda: (
            held_out_final_snapshot
            .account
            .available_cash
            == Decimal("1003.4000")
        ),
    ),
    evaluate_validation_case(
        validation_name=(
            "held_out_final_realized_pnl_expected"
        ),
        validation_callable=lambda: (
            held_out_final_snapshot
            .account
            .realized_pnl
            == Decimal("3.4000")
        ),
    ),
    evaluate_validation_case(
        validation_name=(
            "held_out_final_fees_expected"
        ),
        validation_callable=lambda: (
            held_out_final_snapshot
            .account
            .fees_paid
            == Decimal("0.1100")
        ),
    ),
    evaluate_validation_case(
        validation_name=(
            "held_out_position_settled"
        ),
        validation_callable=lambda: (
            len(
                held_out_final_snapshot.positions
            )
            == 0
        ),
    ),
]

raise_for_failed_validations(
    held_out_validation_results
)

held_out_account_df = pd.DataFrame(
    [
        {
            "available_cash": float(
                held_out_final_snapshot
                .account
                .available_cash
            ),
            "reserved_cash": float(
                held_out_final_snapshot
                .account
                .reserved_cash
            ),
            "realized_pnl": float(
                held_out_final_snapshot
                .account
                .realized_pnl
            ),
            "fees_paid": float(
                held_out_final_snapshot
                .account
                .fees_paid
            ),
            "total_equity": float(
                calculate_total_equity(
                    account=(
                        held_out_final_snapshot.account
                    ),
                    positions=(
                        held_out_final_snapshot.positions
                    ),
                )
            ),
            "last_event_sequence": (
                held_out_final_snapshot
                .last_event_sequence
            ),
        }
    ]
)

display(
    held_out_account_df
)

In [0]:
# Validate that invalid state transitions are rejected

invalid_event_validation_results = [
    evaluate_validation_case(
        validation_name="duplicate_event_id_rejected",
        validation_callable=lambda: apply_event(
            snapshot=development_initial_snapshot,
            event=development_initial_cash_event,
        ),
        expected_error=True,
    ),
    evaluate_validation_case(
        validation_name="insufficient_cash_rejected",
        validation_callable=lambda: apply_event(
            snapshot=development_initial_snapshot,
            event=PortfolioEvent(
                event_id="invalid_insufficient_cash",
                event_sequence=2,
                portfolio_id=(
                    DEVELOPMENT_PORTFOLIO_ID
                ),
                event_ts=(
                    DEVELOPMENT_BASE_TS
                    + timedelta(minutes=1)
                ),
                event_type=(
                    PortfolioEventType.ORDER_PLACED
                ),
                ticker=DEVELOPMENT_TICKER,
                espn_event_id=(
                    DEVELOPMENT_ESPN_EVENT_ID
                ),
                order_id="invalid_cash_order",
                side=OrderSide.BUY,
                quantity=2000,
                price=Decimal("0.99"),
            ),
        ),
        expected_error=True,
    ),
    evaluate_validation_case(
        validation_name="oversell_order_rejected",
        validation_callable=lambda: apply_event(
            snapshot=development_marked_snapshot,
            event=PortfolioEvent(
                event_id="invalid_oversell_order",
                event_sequence=5,
                portfolio_id=(
                    DEVELOPMENT_PORTFOLIO_ID
                ),
                event_ts=(
                    DEVELOPMENT_BASE_TS
                    + timedelta(minutes=5)
                ),
                event_type=(
                    PortfolioEventType.ORDER_PLACED
                ),
                ticker=DEVELOPMENT_TICKER,
                espn_event_id=(
                    DEVELOPMENT_ESPN_EVENT_ID
                ),
                order_id="invalid_oversell_order",
                side=OrderSide.SELL,
                quantity=11,
                price=Decimal("0.54"),
            ),
        ),
        expected_error=True,
    ),
    evaluate_validation_case(
        validation_name="negative_quantity_rejected",
        validation_callable=lambda: validate_event(
            PortfolioEvent(
                event_id="invalid_negative_quantity",
                event_sequence=1,
                portfolio_id=(
                    DEVELOPMENT_PORTFOLIO_ID
                ),
                event_ts=DEVELOPMENT_BASE_TS,
                event_type=(
                    PortfolioEventType.ORDER_PLACED
                ),
                ticker=DEVELOPMENT_TICKER,
                espn_event_id=(
                    DEVELOPMENT_ESPN_EVENT_ID
                ),
                order_id="invalid_negative_order",
                side=OrderSide.BUY,
                quantity=-1,
                price=Decimal("0.40"),
            )
        ),
        expected_error=True,
    ),
    evaluate_validation_case(
        validation_name="naive_timestamp_rejected",
        validation_callable=lambda: validate_event(
            PortfolioEvent(
                event_id="invalid_naive_timestamp",
                event_sequence=1,
                portfolio_id=(
                    DEVELOPMENT_PORTFOLIO_ID
                ),
                event_ts=datetime(
                    2025,
                    9,
                    6,
                    16,
                    0,
                ),
                event_type=(
                    PortfolioEventType.CASH_DEPOSIT
                ),
                cash_amount=STARTING_CASH,
            )
        ),
        expected_error=True,
    ),
    evaluate_validation_case(
        validation_name="invalid_trade_price_rejected",
        validation_callable=lambda: validate_event(
            PortfolioEvent(
                event_id="invalid_trade_price",
                event_sequence=1,
                portfolio_id=(
                    DEVELOPMENT_PORTFOLIO_ID
                ),
                event_ts=DEVELOPMENT_BASE_TS,
                event_type=(
                    PortfolioEventType.ORDER_PLACED
                ),
                ticker=DEVELOPMENT_TICKER,
                espn_event_id=(
                    DEVELOPMENT_ESPN_EVENT_ID
                ),
                order_id="invalid_price_order",
                side=OrderSide.BUY,
                quantity=1,
                price=Decimal("1.01"),
            )
        ),
        expected_error=True,
    ),
    evaluate_validation_case(
        validation_name=(
            "opposing_event_position_rejected"
        ),
        validation_callable=lambda: apply_event(
            snapshot=development_marked_snapshot,
            event=PortfolioEvent(
                event_id="invalid_opposing_order",
                event_sequence=5,
                portfolio_id=(
                    DEVELOPMENT_PORTFOLIO_ID
                ),
                event_ts=(
                    DEVELOPMENT_BASE_TS
                    + timedelta(minutes=5)
                ),
                event_type=(
                    PortfolioEventType.ORDER_PLACED
                ),
                ticker=(
                    DEVELOPMENT_OPPOSING_TICKER
                ),
                espn_event_id=(
                    DEVELOPMENT_ESPN_EVENT_ID
                ),
                order_id="invalid_opposing_order",
                side=OrderSide.BUY,
                quantity=1,
                price=Decimal("0.40"),
            ),
        ),
        expected_error=True,
    ),
    evaluate_validation_case(
        validation_name=(
            "nonincreasing_sequence_rejected"
        ),
        validation_callable=lambda: apply_event(
            snapshot=development_marked_snapshot,
            event=PortfolioEvent(
                event_id="invalid_event_sequence",
                event_sequence=4,
                portfolio_id=(
                    DEVELOPMENT_PORTFOLIO_ID
                ),
                event_ts=(
                    DEVELOPMENT_BASE_TS
                    + timedelta(minutes=5)
                ),
                event_type=(
                    PortfolioEventType.CASH_DEPOSIT
                ),
                cash_amount=Decimal("1.00"),
            ),
        ),
        expected_error=True,
    ),
    evaluate_validation_case(
        validation_name=(
            "backward_event_timestamp_rejected"
        ),
        validation_callable=lambda: apply_event(
            snapshot=development_marked_snapshot,
            event=PortfolioEvent(
                event_id="invalid_event_timestamp",
                event_sequence=5,
                portfolio_id=(
                    DEVELOPMENT_PORTFOLIO_ID
                ),
                event_ts=(
                    DEVELOPMENT_BASE_TS
                    + timedelta(minutes=3)
                ),
                event_type=(
                    PortfolioEventType.CASH_DEPOSIT
                ),
                cash_amount=Decimal("1.00"),
            ),
        ),
        expected_error=True,
    ),
]

raise_for_failed_validations(
    invalid_event_validation_results
)

print(
    "Invalid state transitions were rejected as expected."
)

In [0]:
# Confirm deterministic replay

held_out_replay_one = replay_events(
    snapshot=create_empty_snapshot(
        portfolio_id=HELD_OUT_PORTFOLIO_ID,
        as_of_ts=HELD_OUT_BASE_TS,
    ),
    events=held_out_events,
)

held_out_replay_two = replay_events(
    snapshot=create_empty_snapshot(
        portfolio_id=HELD_OUT_PORTFOLIO_ID,
        as_of_ts=HELD_OUT_BASE_TS,
    ),
    events=held_out_events,
)

determinism_validation_results = [
    evaluate_validation_case(
        validation_name="held_out_replay_deterministic",
        validation_callable=lambda: (
            snapshot_to_dict(
                held_out_replay_one
            )
            == snapshot_to_dict(
                held_out_replay_two
            )
        ),
    )
]

raise_for_failed_validations(
    determinism_validation_results
)

print(
    "Portfolio replay is deterministic."
)

In [0]:
# Consolidate and enforce the complete validation report

all_validation_results = [
    *development_validation_results,
    *held_out_validation_results,
    *invalid_event_validation_results,
    *determinism_validation_results,
]

validation_report_df = pd.DataFrame(
    validation_results_to_records(
        all_validation_results
    )
).sort_values(
    by=[
        "status",
        "validation_name",
    ],
    ascending=[
        True,
        True,
    ],
).reset_index(
    drop=True
)

display(
    validation_report_df
)

raise_for_failed_validations(
    all_validation_results
)

print(
    f"All {len(all_validation_results)} "
    "portfolio validations passed."
)

In [0]:
# Write the approved portfolio-state validation artifact

artifact_created_at_utc = datetime.now(
    timezone.utc
)

portfolio_validation_artifact = {
    "artifact_version": (
        PORTFOLIO_VALIDATION_ARTIFACT_VERSION
    ),
    "portfolio_schema_version": (
        PORTFOLIO_SCHEMA_VERSION
    ),
    "created_at_utc": (
        artifact_created_at_utc.isoformat()
    ),
    "portfolio_configuration": (
        build_portfolio_configuration_snapshot()
    ),
    "supported_event_types": [
        event_type.value
        for event_type in PortfolioEventType
    ],
    "required_invariants": list(
        REQUIRED_PORTFOLIO_INVARIANTS
    ),
    "validation_summary": {
        "validation_count": len(
            all_validation_results
        ),
        "passed_count": sum(
            result.status == "passed"
            for result in all_validation_results
        ),
        "failed_count": sum(
            result.status != "passed"
            for result in all_validation_results
        ),
    },
    "validation_results": (
        validation_results_to_records(
            all_validation_results
        )
    ),
    "development_final_snapshot": (
        snapshot_to_dict(
            development_final_snapshot
        )
    ),
    "held_out_final_snapshot": (
        snapshot_to_dict(
            held_out_final_snapshot
        )
    ),
    "source_metadata": {
        "python_version": (
            platform.python_version()
        ),
        "repository_root": str(
            REPOSITORY_ROOT
        ),
        "source_modules": [
            "src.portfolio.models",
            "src.portfolio.accounting",
            "src.portfolio.transitions",
            "src.portfolio.validation",
            "src.portfolio.store",
            "src.portfolio.portfolio_config",
        ],
    },
}

write_json_artifact(
    artifact_path=(
        PORTFOLIO_VALIDATION_ARTIFACT_PATH
    ),
    payload=portfolio_validation_artifact,
)

print(
    "Portfolio validation artifact written to: "
    f"{PORTFOLIO_VALIDATION_ARTIFACT_PATH}"
)

In [0]:
# Reload and validate the approved artifact

reloaded_validation_artifact = (
    read_json_artifact(
        PORTFOLIO_VALIDATION_ARTIFACT_PATH
    )
)

required_artifact_keys = {
    "artifact_version",
    "portfolio_schema_version",
    "created_at_utc",
    "portfolio_configuration",
    "supported_event_types",
    "required_invariants",
    "validation_summary",
    "validation_results",
    "source_metadata",
}

missing_artifact_keys = (
    required_artifact_keys
    - set(
        reloaded_validation_artifact
    )
)

if missing_artifact_keys:
    raise ValueError(
        "Portfolio validation artifact is missing "
        "required keys. "
        f"Missing: {sorted(missing_artifact_keys)}"
    )

if (
    reloaded_validation_artifact[
        "artifact_version"
    ]
    != PORTFOLIO_VALIDATION_ARTIFACT_VERSION
):
    raise ValueError(
        "Portfolio validation artifact version does not "
        "match the current configuration."
    )

if (
    reloaded_validation_artifact[
        "portfolio_schema_version"
    ]
    != PORTFOLIO_SCHEMA_VERSION
):
    raise ValueError(
        "Portfolio validation artifact schema version does "
        "not match the current configuration."
    )

if (
    reloaded_validation_artifact[
        "portfolio_configuration"
    ]
    != build_portfolio_configuration_snapshot()
):
    raise ValueError(
        "Portfolio validation artifact configuration does "
        "not match the current portfolio configuration."
    )

if (
    reloaded_validation_artifact[
        "validation_summary"
    ][
        "failed_count"
    ]
    != 0
):
    raise ValueError(
        "Portfolio validation artifact contains failed "
        "validations."
    )

print(
    "Portfolio validation artifact passed the reload "
    "smoke test and is ready for the deployment notebook."
)